# Intelligence & Win Alignment — CDFI Fund CY 2024-2025 Scoring Framework

This notebook demonstrates the full intelligence pipeline using the **CDFI Fund's published CY 2024-2025 Review Process** as the scoring framework.

The story arc:

| Stage | What we build | CDFI Fund tier |
|---|---|---|
| 1 | LIC-only pipeline, no CDE track record | **Not Qualified** — fails section minimums |
| 2 | Deep-distress pipeline + moderate CDE attributes | **Highly Qualified** — meets 40/50 section minimums, 85+ aggregate |
| 3 | Optimized pipeline + strong CDE attributes | **Top Tier** — 45/50 in both sections, 95+ aggregate |

---

## CDFI Fund Scoring Framework Summary

| Section | Max | Minimum for Highly Qualified |
|---|---|---|
| Business Strategy | 50 pts | ≥ 40 pts |
| Community Outcomes | 50 pts | ≥ 40 pts |
| **Aggregate Base** | **100 pts** | **≥ 85 pts** |
| Priority Points (bonus) | +10 pts | n/a (not gating) |

> **Disclosure:** Sub-score weights within each section are this tool's interpretation of the CDFI Fund's published Review Process document. The CDFI Fund does not publish exact point values for individual sub-criteria. Scores reflect self-assessment alignment, not a prediction of award outcome.

In [ ]:
import sys
sys.path.insert(0, '..')

from nmtcapp.core.pipeline import Pipeline, PipelineProject
from nmtcapp.core.cde import CDEProfile
from nmtcapp.core.application import Application
from nmtcapp.intelligence.win_probability import WinProbabilityModel
from nmtcapp.intelligence.pipeline_analyzer import PipelineAnalyzer
from nmtcapp.intelligence.recommendations import RecommendationEngine
from nmtcapp.intelligence.benchmarks import HistoricalBenchmarks
from nmtcapp.data.benchmark_thresholds import (
    HIGHLY_QUALIFIED_AGGREGATE_MIN, HIGHLY_QUALIFIED_SECTION_MIN,
    TOP_TIER_AGGREGATE_MIN, TOP_TIER_SECTION_MIN,
)

---
## Part 1 — Weak Pipeline → Not Qualified

We start with a pipeline that has two common problems:

- **All projects are standard-LIC** — no deep or severe distress targeting
- **No CDE track record data** — product flexibility and track record scores default to 0

This pipeline fails the Community Outcomes section minimum (< 40/50), which means it does **not advance to Phase 2** regardless of its aggregate score.

In [ ]:
# Build a weak pipeline: 8 projects, all LIC-only distress, single state/sector
weak = Pipeline()
for i in range(8):
    p = PipelineProject(
        project_id=f"W-{i:03d}",
        project_name=f"Chicago Community Facility {i+1}",
        qalicb_name=f"IL QALICB {i+1}",
        address=f"{100+i} S Michigan Ave",
        city="Chicago",
        state="IL",
        sector="community_facility",
        project_type="real_estate",
        total_project_cost=8_000_000,
        qei_request=5_000_000,
        qlici_amount=5_000_000,
        expected_jobs_created=8,
        is_nmtc_eligible=True,
        distress_level="lic",        # LIC-only — no deep/severe distress
        is_native_area=False,
        is_high_migration_rural=False,
        is_opportunity_zone=False,
    )
    weak.add(p)

result_weak = PipelineAnalyzer().analyze(weak)
pr = result_weak  # PipelineAnalysisResult is the pipeline result directly

print(f"Projects:            {pr.total_projects}")
print(f"Deep/Severe distress:{pr.distress_breakdown['pct_deep_or_severe']:.0%} of QEI")
print(f"Deep distress only:  {pr.distress_breakdown.get('pct_deep', 0):.0%} of QEI")
print(f"NMTC eligibility:    {pr.eligibility_pct:.0%}")
print(f"States:              {pr.geographic_diversity['states_count']}")
print(f"Sectors:             {pr.sector_mix['sectors_represented']}")

In [ ]:
# Score against the CDFI Fund framework — no CDE attributes provided
# (product flexibility, track record, board composition all score 0)
score_weak = WinProbabilityModel().score(
    result_weak, 40_000_000, cde_attributes={}
)

print("══ CDFI Fund CY 2024-2025 Score — Weak Pipeline ══")
print(f"Tier:                   {score_weak.tier}")
print(f"Aggregate Base Score:   {score_weak.aggregate_base_score}/100  "
      f"(gate: {HIGHLY_QUALIFIED_AGGREGATE_MIN}+)")
print(f"Business Strategy:      {score_weak.business_strategy['section_total']}/50  "
      f"(gate: {HIGHLY_QUALIFIED_SECTION_MIN}+)")
print(f"Community Outcomes:     {score_weak.community_outcomes['section_total']}/50  "
      f"(gate: {HIGHLY_QUALIFIED_SECTION_MIN}+)")
print(f"Priority Points:        {score_weak.priority_points['section_total']}/10")
print()
if score_weak.tier_gating_notes:
    print("Gating failures (application does not advance to Phase 2):")
    for note in score_weak.tier_gating_notes:
        print(f"  ⚠️  {note}")

In [ ]:
# Drill into the sub-scores to understand exactly where the gaps are
bs = score_weak.business_strategy
co = score_weak.community_outcomes

print("Business Strategy breakdown:")
print(f"  Product Flexibility:      {bs.get('product_flexibility', 0):5.1f} / 10")
print(f"  Pipeline Credibility:     {bs.get('pipeline_credibility', 0):5.1f} / 15")
print(f"  Track Record Strength:    {bs.get('track_record_strength', 0):5.1f} / 15")
print(f"  Track Record Alignment:   {bs.get('track_record_alignment', 0):5.1f} / 10")
print(f"  ── Section total:         {bs['section_total']:5.1f} / 50")
print()
print("Community Outcomes breakdown:")
print(f"  Higher Distress Targeting:{co.get('higher_distress_targeting', 0):5.1f} / 15")
print(f"  Deep Distress Commitment: {co.get('deep_distress_commitment', 0):5.1f} / 10")
print(f"  Special Targeting:        {co.get('special_targeting', 0):5.1f} /  5")
print(f"  Community Outcomes Qual.: {co.get('community_outcomes_quality', 0):5.1f} / 10")
print(f"  Community Accountability: {co.get('community_accountability', 0):5.1f} / 10")
print(f"  ── Section total:         {co['section_total']:5.1f} / 50")

**Diagnosis:** The Community Outcomes section scores near 0 because:
- Higher Distress Targeting = 0 (0% of QEI in deep/severe distress; threshold is 85%)
- Deep Distress Commitment = 0 (0% in CDFI Fund Deep Distress areas; threshold is 20%)
- Community Accountability = 0 (no board composition data provided)

The Business Strategy section is also weak because no CDE product or track record data was provided.

**To advance to Phase 2**, both sections must reach ≥ 40/50 AND the aggregate must reach 85+.

---
## Part 2 — Benchmark Against Historical Winners

The `HistoricalBenchmarks` module compares pipeline metrics against patterns observed in CY2020–2024 NMTC award winners.

> **Note:** This is separate from the CDFI Fund score — benchmarks reflect winner patterns, while the alignment score measures published criteria.

In [ ]:
bc_weak = HistoricalBenchmarks().compare(result_weak, 40_000_000)
print(bc_weak.summary())

---
## Part 3 — Improved Pipeline → Highly Qualified

To reach **Highly Qualified**, both sections must score ≥ 40/50 and the aggregate must be ≥ 85/100.

Changes made:
- Pipeline: Use `Pipeline.sample()` — 20 projects across 20 states, 87% deep/severe distress
- CDE attributes: Moderate product flexibility, 3 prior NMTC awards, partial board representation

This should land in the **Highly Qualified** band (~85–94/100).

In [ ]:
# Improved pipeline: Pipeline.sample() ships with deep/severe distress and geographic diversity
improved = Pipeline.sample(n=20)
result_improved = PipelineAnalyzer().analyze(improved)
pr2 = result_improved  # PipelineAnalysisResult is the pipeline result directly

print(f"Projects:            {pr2.total_projects}")
print(f"Deep/Severe distress:{pr2.distress_breakdown['pct_deep_or_severe']:.0%} of QEI")
print(f"Deep distress only:  {pr2.distress_breakdown.get('pct_deep', 0):.0%} of QEI")
print(f"States:              {pr2.geographic_diversity['states_count']}")
print(f"Sectors:             {pr2.sector_mix['sectors_represented']}")
print(f"Jobs/$MM QEI:        {pr2.aggregate_impact['jobs_per_million_qei']:.1f}")

In [ ]:
# Moderate CDE attributes — representative of a maturing CDE
# These values are passed separately from the pipeline because the CDFI Fund evaluates
# CDE-level attributes (product terms, governance, track record) through narrative review.
moderate_cde = {
    # Business Strategy: Product Flexibility
    "products_below_market_pct": 0.42,       # 42% below-market (just under the 50% threshold)
    "products_flexible_indicia_count": 5,     # meets the 5-indicia alternative threshold
    # Business Strategy: Track Record
    "prior_award_count": 3,                   # 3 prior NMTC awards
    "years_in_operation": 7,                  # 7 years
    "has_own_capital_at_risk": False,
    "pipeline_pct_identified": 0.83,          # 83% pipeline identified
    "track_record_pipeline_alignment_pct": 0.76,  # 76% alignment (≥70% threshold)
    "track_record_deployment_pct": 0.80,      # 80% deployment (below 90% threshold)
    # Community Outcomes: Quality & Accountability
    "has_quantified_outcomes": True,
    "has_third_party_validation": True,       # third-party validated → 9/10 outcomes quality
    "lic_board_representation_pct": 0.44,     # 44% LIC board representation
    "has_community_engagement_track_record": True,
    "pct_persistent_poverty": 0.31,           # 31% in persistent poverty counties
    # Priority Points (weak/absent — will be near 0)
    "dbc_focus_years": 0,
    "dbc_dollar_volume_pct": 0.0,
    "unrelated_entities_pct": 0.82,           # 82% unrelated (below 90% threshold)
}

score_improved = WinProbabilityModel().score(
    result_improved, 55_000_000, cde_attributes=moderate_cde
)

print("══ CDFI Fund CY 2024-2025 Score — Improved Pipeline ══")
print(f"Tier:                   {score_improved.tier}")
print(f"Aggregate Base Score:   {score_improved.aggregate_base_score}/100  "
      f"(gate: {HIGHLY_QUALIFIED_AGGREGATE_MIN}+)")
print(f"With Priority Points:   {score_improved.aggregate_with_priority}/110")
print()
bs2 = score_improved.business_strategy
co2 = score_improved.community_outcomes
pp2 = score_improved.priority_points
print(f"Business Strategy:      {bs2['section_total']}/50  "
      f"({'✓ meets' if bs2['section_total'] >= HIGHLY_QUALIFIED_SECTION_MIN else '✗ below'} "
      f"{HIGHLY_QUALIFIED_SECTION_MIN} minimum)")
print(f"Community Outcomes:     {co2['section_total']}/50  "
      f"({'✓ meets' if co2['section_total'] >= HIGHLY_QUALIFIED_SECTION_MIN else '✗ below'} "
      f"{HIGHLY_QUALIFIED_SECTION_MIN} minimum)")
print(f"Priority Points:        {pp2['section_total']}/10")

if score_improved.tier_gating_notes:
    print()
    print("Gating notes:")
    for note in score_improved.tier_gating_notes:
        print(f"  ⚠️  {note}")

In [ ]:
# Compare section-by-section: weak vs. improved
def _section_bar(score, max_score=10):
    filled = int(round(score / max_score * 10))
    return '█' * filled + '░' * (10 - filled)

print(f"{'Sub-criterion':<32} {'WEAK':>6} {'IMPROVED':>9}  {'Max':>4}")
print("-" * 58)

sub_criteria = [
    ("BS: Product Flexibility",     "product_flexibility",       10, "business_strategy"),
    ("BS: Pipeline Credibility",    "pipeline_credibility",      15, "business_strategy"),
    ("BS: Track Record Strength",   "track_record_strength",     15, "business_strategy"),
    ("BS: Track Record Alignment",  "track_record_alignment",    10, "business_strategy"),
    ("CO: Higher Distress",         "higher_distress_targeting", 15, "community_outcomes"),
    ("CO: Deep Distress",           "deep_distress_commitment",  10, "community_outcomes"),
    ("CO: Special Targeting",       "special_targeting",          5, "community_outcomes"),
    ("CO: Outcomes Quality",        "community_outcomes_quality",10, "community_outcomes"),
    ("CO: Accountability",          "community_accountability",  10, "community_outcomes"),
]

for label, key, max_pts, section in sub_criteria:
    w_score = getattr(score_weak, section).get(key, 0)
    i_score = getattr(score_improved, section).get(key, 0)
    delta = i_score - w_score
    arrow = "↑" if delta > 0 else ("─" if delta == 0 else "↓")
    print(f"{label:<32} {w_score:>5.1f} → {i_score:>5.1f}  {arrow} {delta:+.1f}   / {max_pts}")

print("-" * 58)
print(f"{'SECTION TOTALS':<32} "
      f"{score_weak.aggregate_base_score:>5}   {score_improved.aggregate_base_score:>5}          / 100")

---
## Part 4 — Recommendations

The `RecommendationEngine` generates prioritized, quantified actions citing specific CDFI Fund Review Process sections.

In [ ]:
bc_improved = HistoricalBenchmarks().compare(result_improved, 55_000_000)
recs = RecommendationEngine().recommend(
    result_improved, bc_improved, score_improved
)

print(f"Overall assessment: {recs.overall_assessment}")
print(f"Total recommendations: {len(recs.recommendations)}")
print(f"  Critical: {sum(1 for r in recs.recommendations if r.priority == 'critical')}")
print(f"  High:     {sum(1 for r in recs.recommendations if r.priority == 'high')}")
print(f"  Medium:   {sum(1 for r in recs.recommendations if r.priority == 'medium')}")

In [ ]:
# Display all recommendations with CDFI Fund citations
for priority_label, priority in [("CRITICAL", "critical"), ("HIGH", "high"), ("MEDIUM", "medium")]:
    priority_recs = [r for r in recs.recommendations if r.priority == priority]
    if not priority_recs:
        continue
    print(f"\n[{priority_label}]")
    print("─" * 70)
    for r in priority_recs:
        print(f"Category:  {r.category}")
        print(f"Finding:   {r.finding}")
        print(f"Action:    {r.action}")
        print(f"Estimate:  {r.quantified_improvement}")
        if r.citation:
            print(f"Citation:  {r.citation}")
        print()

---
## Part 5 — Optimized Application → Top Tier

**Top Tier** requires:
- Aggregate base score ≥ 95/100
- Both sections ≥ 45/50

This requires strong performance across every sub-criterion:
- Business Strategy: ≥ 50% below-market products OR ≥ 5 indicia, high pipeline identification, strong track record
- Community Outcomes: ≥ 85% deep/severe distress, ≥ 20% deep distress, third-party validated outcomes, strong community accountability
- Priority Points: DBC track record + unrelated entities boost the aggregate to 95+

In [ ]:
# Strong CDE attributes — representative of a leading, established CDE
strong_cde = {
    # Business Strategy: Product Flexibility — exceeds both thresholds
    "products_below_market_pct": 0.60,        # 60% below-market (exceeds 50% threshold)
    "products_flexible_indicia_count": 6,      # 6 flexible indicia (exceeds 5 threshold)
    # Business Strategy: Pipeline & Track Record
    "pipeline_pct_identified": 0.95,           # 95% pipeline identified
    "prior_award_count": 4,                    # 4 prior NMTC awards
    "years_in_operation": 8,                   # 8 years in operation
    "has_own_capital_at_risk": True,           # +3 track record bonus
    "track_record_pipeline_alignment_pct": 0.85,  # 85% alignment (exceeds 70% threshold)
    "track_record_deployment_pct": 0.95,       # 95% deployment (exceeds 90% threshold)
    # Community Outcomes: Quality & Accountability
    "has_quantified_outcomes": True,
    "has_third_party_validation": True,        # 9/10 community outcomes quality
    "lic_board_representation_pct": 0.50,      # 50% LIC board representation
    "has_community_engagement_track_record": True,
    "pct_persistent_poverty": 0.40,            # 40% in persistent poverty counties
    # Priority Points: Both criteria met
    "dbc_focus_years": 6,                      # 6 years ≥ 5-year threshold
    "dbc_dollar_volume_pct": 0.80,             # 80% ≥ 70% threshold
    "unrelated_entities_pct": 0.95,            # 95% ≥ 90% threshold → full 5 pts
}

score_top = WinProbabilityModel().score(
    result_improved, 55_000_000, cde_attributes=strong_cde
)

print("══ CDFI Fund CY 2024-2025 Score — Optimized Application ══")
print(f"Tier:                   {score_top.tier}")
print(f"Aggregate Base Score:   {score_top.aggregate_base_score}/100")
print(f"With Priority Points:   {score_top.aggregate_with_priority}/110")
print()
bs3 = score_top.business_strategy
co3 = score_top.community_outcomes
pp3 = score_top.priority_points
hq_gate = HIGHLY_QUALIFIED_SECTION_MIN
tt_gate = TOP_TIER_SECTION_MIN
print(f"Business Strategy:      {bs3['section_total']}/50  "
      f"({'✓ Top Tier' if bs3['section_total'] >= tt_gate else '✓ HQ' if bs3['section_total'] >= hq_gate else '✗ below min'})")
print(f"Community Outcomes:     {co3['section_total']}/50  "
      f"({'✓ Top Tier' if co3['section_total'] >= tt_gate else '✓ HQ' if co3['section_total'] >= hq_gate else '✗ below min'})")
print(f"Priority Points:        {pp3['section_total']}/10")
print(f"  DBC Track Record:       {pp3.get('dbc_track_record', 0):.1f}/5")
print(f"  Unrelated Entities:     {pp3.get('unrelated_entities', 0):.1f}/5")

if score_top.tier_gating_notes:
    for note in score_top.tier_gating_notes:
        print(f"  ⚠️  {note}")

In [ ]:
# Three-way comparison: Not Qualified → Highly Qualified → Top Tier
print(f"{'Metric':<32} {'Not Qualified':>14} {'Highly Qual.':>13} {'Top Tier':>9}")
print("─" * 72)

rows = [
    ("Tier",                       score_weak.tier,        score_improved.tier,        score_top.tier),
    ("Aggregate Base Score",        score_weak.aggregate_base_score,
                                    score_improved.aggregate_base_score,
                                    score_top.aggregate_base_score),
    ("With Priority Points",        score_weak.aggregate_with_priority,
                                    score_improved.aggregate_with_priority,
                                    score_top.aggregate_with_priority),
    ("Business Strategy /50",       score_weak.business_strategy['section_total'],
                                    score_improved.business_strategy['section_total'],
                                    score_top.business_strategy['section_total']),
    ("Community Outcomes /50",      score_weak.community_outcomes['section_total'],
                                    score_improved.community_outcomes['section_total'],
                                    score_top.community_outcomes['section_total']),
    ("Priority Points /10",         score_weak.priority_points['section_total'],
                                    score_improved.priority_points['section_total'],
                                    score_top.priority_points['section_total']),
]

for label, v1, v2, v3 in rows:
    print(f"{label:<32} {str(v1):>14} {str(v2):>13} {str(v3):>9}")

---
## Part 6 — Pipeline Optimizer

The optimizer selects a subset of projects from the improved pipeline that maximizes alignment with historical winner patterns, subject to QEI budget and diversity constraints.

> **Note:** The optimizer targets historical winner pattern alignment (the benchmark score), not the CDFI Fund section scores directly. Use it to improve pipeline composition for Phase 2 review readiness.

In [ ]:
from nmtcapp.optimizer import OptimizationConstraints, PipelineOptimizer

constraints = OptimizationConstraints(
    min_total_qei=40_000_000,
    max_total_qei=55_000_000,
    min_projects=8,
    min_states=6,
    required_sectors=["healthcare"],
)

opt_result = PipelineOptimizer(max_iterations=200).optimize(
    improved, constraints, 55_000_000
)
print(opt_result.summary())

In [ ]:
# Projects selected by the optimizer
print(f"{'ID':<10} {'State':>5} {'Sector':<22} {'QEI':>12} {'Jobs':>6}  Distress")
print("─" * 72)
for p in opt_result.selected_projects:
    print(f"{p.project_id:<10} {p.state:>5} {p.sector:<22} "
          f"${p.qei_request:>10,.0f} {p.expected_jobs_created:>6}  {p.distress_level}")

total_qei = sum(p.qei_request for p in opt_result.selected_projects)
total_jobs = sum(p.expected_jobs_created for p in opt_result.selected_projects)
states_sel = len({p.state for p in opt_result.selected_projects})
deep_sel = sum(p.qei_request for p in opt_result.selected_projects if p.distress_level in ('deep','severe'))
print("─" * 72)
print(f"Total QEI: ${total_qei:,.0f}  |  States: {states_sel}  |  "
      f"Jobs/$MM: {total_jobs/(total_qei/1_000_000):.1f}  |  "
      f"Deep/Severe: {deep_sel/total_qei:.0%}")

---
## Part 7 — Full `Application` Interface

All of the above is also accessible through the `Application` facade, which reads CDE attributes from `CDEProfile.extra` automatically.

In [ ]:
# CDEProfile.sample() includes Riverbend Community Capital's scoring attributes in .extra
# These attributes drive the CDFI Fund section scores automatically.
cde = CDEProfile.sample()
print(f"CDE: {cde.name}")
print(f"Scoring attributes in .extra: {list(cde.extra.keys())}")

In [ ]:
app = Application(cde=cde, requested_allocation=55_000_000)
app.add_pipeline(improved)

# score_win_probability() passes cde.extra → cde_attributes automatically
app_score = app.score_win_probability()

print(f"Tier:                  {app_score.tier}")
print(f"Aggregate Base Score:  {app_score.aggregate_base_score}/100")
print(f"With Priority Points:  {app_score.aggregate_with_priority}/110")
print(f"Business Strategy:     {app_score.business_strategy['section_total']}/50")
print(f"Community Outcomes:    {app_score.community_outcomes['section_total']}/50")
print(f"Priority Points:       {app_score.priority_points['section_total']}/10")
print()
print("Phase 2 flags (informational — not scored):")
for flag, value in app_score.phase2_flags.items():
    status = "⚠️" if value else "✓"
    print(f"  {status}  {flag.replace('_', ' ')}: {value}")

In [ ]:
# Recommendations via Application.recommendations() — uses the last computed win score
app_recs = app.recommendations()
print(app_recs.summary())

---
## Summary

This notebook demonstrated the full intelligence workflow aligned to the **CDFI Fund's published CY 2024-2025 Review Process**:

| Step | Tool | What You Get |
|---|---|---|
| Pipeline scoring | `WinProbabilityModel().score()` | CDFI Fund tier + section scores (BS/CO/PP) |
| Benchmark | `HistoricalBenchmarks().compare()` | 9-metric comparison vs. CY2020-2024 winners |
| Recommendations | `RecommendationEngine().recommend()` | Prioritized actions with CDFI Fund citations |
| Optimization | `PipelineOptimizer().optimize()` | Max-alignment project subset |
| Full facade | `Application.score_win_probability()` | All of the above from a single object |

### The three tiers

| Tier | Aggregate | Both sections | Outcome |
|---|---|---|---|
| Not Qualified | < 85 | Either < 40/50 | Does not advance to Phase 2 |
| Highly Qualified | 85–94 | Both ≥ 40/50 | Phase 2 reviewed; award depends on ranking |
| Top Tier | ≥ 95 | Both ≥ 45/50 | High probability of Phase 2 advancement |

> **Disclosure:** Scores reflect self-assessment alignment with the CDFI Fund's *published* criteria. The CDFI Fund's actual scoring rubric is proprietary; sub-score weights are this tool's best-effort interpretation. This tool does not predict award outcomes.
>
> **Source:** CDFI Fund. *CY 2024-2025 NMTC Allocation Application Review Process.* U.S. Department of the Treasury.